In [1]:
import numpy as np
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# ============================================================
# WEEK 3 ASSIGNMENT, Q1 - 2-Layer CNN on CIFAR-10
#
# Your job: implement the four functions marked TODO.
#   - conv_forward
#   - conv_backward
#   - maxpool_forward (already provided)
#   - maxpool_backward (already provided)
#
# Everything else (ReLU, FC, loss, training) is provided.
# NumPy only inside the four functions — no PyTorch autograd.
#
# Architecture:
#   Conv(3→8, 3x3, pad=1) → ReLU → MaxPool(2x2)
#   Conv(8→16, 3x3, pad=1) → ReLU → MaxPool(2x2)
#   Flatten → FC(1024→10) → Softmax
#
# Target: ~30% test accuracy after 20 epochs (CPU is fine).
# ============================================================


# ------------------------------------------------------------
# Data loading  (nothing to change here)
# ------------------------------------------------------------

def get_dataloaders(batch_size=64):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    train_set = torchvision.datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform)
    test_set = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform)
    # Small subset so it runs in reasonable time on CPU
    train_set = torch.utils.data.Subset(train_set, range(3000))
    test_set  = torch.utils.data.Subset(test_set,  range(500))
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False)
    return train_loader, test_loader


# ------------------------------------------------------------
# TODO 1 — Conv forward pass
# ------------------------------------------------------------

def conv_forward(X, W, b, stride=1, pad=1):
    """
    Forward pass for a convolutional layer.

    Args:
        X  : numpy array, shape (N, C_in, H, W)
        W  : filters,     shape (C_out, C_in, kH, kW)
        b  : bias,        shape (C_out,)

    Returns:
        out   : numpy array, shape (N, C_out, H_out, W_out)
        cache : tuple of everything needed for the backward pass
    """
    N, C_in, H, W_in = X.shape
    C_out, _, kH, kW = W.shape

    H_out = (H + 2 * pad - kH) // stride + 1
    W_out = (W_in + 2 * pad - kW) // stride + 1

    # Pad X spatially (only H and W dimensions, with zeros)
    X_pad = np.pad(X, ((0, 0), (0, 0), (pad, pad), (pad, pad)), mode='constant')

    out = np.zeros((N, C_out, H_out, W_out))

    for h in range(H_out):
        for w in range(W_out):
            # patch: (N, C_in, kH, kW)
            patch = X_pad[:, :, h * stride:h * stride + kH, w * stride:w * stride + kW]
            # for each filter f, sum over (C_in, kH, kW) -> (N, C_out)
            out[:, :, h, w] = np.einsum('ncij,fcij->nf', patch, W) + b

    cache = (X_pad, W, b, stride, pad)
    return out, cache


# ------------------------------------------------------------
# TODO 2 — Conv backward pass
# ------------------------------------------------------------

def conv_backward(dout, cache):
    """
    Backward pass for the convolutional layer.

    Args:
        dout  : upstream gradient, shape (N, C_out, H_out, W_out)
        cache : from conv_forward

    Returns:
        dX : shape (N, C_in, H, W)
        dW : shape (C_out, C_in, kH, kW)
        db : shape (C_out,)
    """
    X_pad, W, b, stride, pad = cache
    N, C_in, H_pad, W_pad = X_pad.shape
    C_out, _, kH, kW = W.shape
    _, _, H_out, W_out = dout.shape

    dX_pad = np.zeros_like(X_pad)
    dW     = np.zeros_like(W)

    # db[f] = sum of dout[:, f, :, :] over all N, H_out, W_out
    db = dout.sum(axis=(0, 2, 3))

    for h in range(H_out):
        for w in range(W_out):
            patch = X_pad[:, :, h * stride:h * stride + kH, w * stride:w * stride + kW]
            dW += np.einsum('nf,ncij->fcij', dout[:, :, h, w], patch)
            dX_pad[:, :, h * stride:h * stride + kH, w * stride:w * stride + kW] += \
                np.einsum('nf,fcij->ncij', dout[:, :, h, w], W)

    dX = dX_pad[:, :, pad:-pad, pad:-pad] if pad > 0 else dX_pad
    return dX, dW, db


# ------------------------------------------------------------
# maxpool_forward
# ------------------------------------------------------------

def maxpool_forward(X, pool_size=2, stride=2):
    N, C, H, W = X.shape
    H_out = (H - pool_size) // stride + 1
    W_out = (W - pool_size) // stride + 1
    out = np.zeros((N, C, H_out, W_out))

    for h in range(H_out):
        for w in range(W_out):
            # Extract the pooling window for all N and C at once.
            window = X[:, :,
                       h * stride : h * stride + pool_size,
                       w * stride : w * stride + pool_size]
            # Take max over the spatial extent of the window (axes 2 and 3).
            out[:, :, h, w] = window.max(axis=(2, 3))

    cache = (X, pool_size, stride)
    return out, cache


# ------------------------------------------------------------
# maxpool_backward
# ------------------------------------------------------------

def maxpool_backward(dout, cache):
    X, pool_size, stride = cache
    N, C, H, W = X.shape
    _, _, H_out, W_out = dout.shape
    dX = np.zeros_like(X)

    for h in range(H_out):
        for w in range(W_out):
            window = X[:, :,
                       h * stride : h * stride + pool_size,
                       w * stride : w * stride + pool_size]

            # Find the flat index of the max within each (pool_size x pool_size) window.
            # Reshape to (N, C, pool_size*pool_size) so argmax gives a scalar per (n, c).
            flat      = window.reshape(N, C, -1)
            max_idx   = flat.argmax(axis=2)            # shape: (N, C)

            # Convert flat index back to 2D (row, col) within the window.
            max_row   = max_idx // pool_size
            max_col   = max_idx %  pool_size

            # Build index arrays to scatter dout into dX.
            n_idx = np.arange(N)[:, None]              # (N, 1)
            c_idx = np.arange(C)[None, :]              # (1, C)

            dX[n_idx, c_idx,
               h * stride + max_row,
               w * stride + max_col] += dout[:, :, h, w]

    return dX


# ------------------------------------------------------------
# Provided layers  (do not modify)
# ------------------------------------------------------------

def relu_forward(X):
    return np.maximum(0, X), X

def relu_backward(dout, cache):
    return dout * (cache > 0)

def fc_forward(X, W, b):
    return X @ W + b, (X, W)

def fc_backward(dout, cache):
    X, W = cache
    return dout @ W.T, X.T @ dout, dout.sum(axis=0)

def softmax_loss(logits, y):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp     = np.exp(shifted)
    probs   = exp / exp.sum(axis=1, keepdims=True)
    N       = logits.shape[0]
    loss    = -np.log(probs[np.arange(N), y] + 1e-8).mean()
    dlogits = probs.copy()
    dlogits[np.arange(N), y] -= 1
    dlogits /= N
    return loss, dlogits


# ------------------------------------------------------------
# Parameter initialisation  (do not modify)
# ------------------------------------------------------------

def init_params():
    p = {}
    p['W1'] = np.random.randn(8,  3,  3, 3).astype(np.float32) * np.sqrt(2 / (3*3*3))
    p['b1'] = np.zeros(8,  dtype=np.float32)
    p['W2'] = np.random.randn(16, 8,  3, 3).astype(np.float32) * np.sqrt(2 / (3*3*8))
    p['b2'] = np.zeros(16, dtype=np.float32)
    # After conv1+pool: 32→16, after conv2+pool: 16→8. Flat size: 16*8*8 = 1024
    p['W3'] = np.random.randn(1024, 10).astype(np.float32) * np.sqrt(2 / 1024)
    p['b3'] = np.zeros(10, dtype=np.float32)
    return p


# ------------------------------------------------------------
# Forward + backward  (do not modify)
# ------------------------------------------------------------

def forward_backward(X, y, params):
    # Block 1
    c1, cc1 = conv_forward(X, params['W1'], params['b1'])
    r1, cr1 = relu_forward(c1)
    p1, cp1 = maxpool_forward(r1)
    # Block 2
    c2, cc2 = conv_forward(p1, params['W2'], params['b2'])
    r2, cr2 = relu_forward(c2)
    p2, cp2 = maxpool_forward(r2)
    # FC
    flat         = p2.reshape(X.shape[0], -1)
    logits, cfc  = fc_forward(flat, params['W3'], params['b3'])
    loss, dlogits = softmax_loss(logits, y)

    # Backward
    dflat, dW3, db3 = fc_backward(dlogits, cfc)
    dp2  = dflat.reshape(p2.shape)
    dr2  = maxpool_backward(dp2, cp2)
    dc2  = relu_backward(dr2, cr2)
    dp1, dW2, db2 = conv_backward(dc2, cc2)
    dr1  = maxpool_backward(dp1, cp1)
    dc1  = relu_backward(dr1, cr1)
    _,   dW1, db1 = conv_backward(dc1, cc1)

    grads = {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2, 'W3': dW3, 'b3': db3}
    return loss, grads


# ------------------------------------------------------------
# Training loop  (do not modify)
# ------------------------------------------------------------

def train(lr=1e-3, batch_size=64, epochs=20):
    train_loader, test_loader = get_dataloaders(batch_size)
    params = init_params()

    for epoch in range(epochs):
        total_loss, n_batches = 0.0, 0
        for Xb, yb in train_loader:
            Xb = Xb.numpy()
            yb = yb.numpy()
            loss, grads = forward_backward(Xb, yb, params)
            for k in params:
                params[k] -= lr * grads[k]
            total_loss += loss
            n_batches  += 1

        acc = evaluate(test_loader, params)
        print(f"Epoch {epoch+1}/{epochs}  "
              f"loss: {total_loss/n_batches:.4f}  "
              f"test acc: {acc:.3f}")

    return params


def evaluate(loader, params):
    correct, total = 0, 0
    for Xb, yb in loader:
        Xb, yb = Xb.numpy(), yb.numpy()
        c1, _ = conv_forward(Xb, params['W1'], params['b1'])
        r1, _ = relu_forward(c1)
        p1, _ = maxpool_forward(r1)
        c2, _ = conv_forward(p1, params['W2'], params['b2'])
        r2, _ = relu_forward(c2)
        p2, _ = maxpool_forward(r2)
        flat      = p2.reshape(Xb.shape[0], -1)
        logits, _ = fc_forward(flat, params['W3'], params['b3'])
        correct  += (logits.argmax(axis=1) == yb).sum()
        total    += len(yb)
    return correct / total


if __name__ == '__main__':
    train()


100%|██████████| 170M/170M [00:08<00:00, 20.4MB/s] 


Epoch 1/20  loss: 2.4530  test acc: 0.110
Epoch 2/20  loss: 2.3537  test acc: 0.134
Epoch 3/20  loss: 2.3168  test acc: 0.178
Epoch 4/20  loss: 2.2918  test acc: 0.186
Epoch 5/20  loss: 2.2695  test acc: 0.188
Epoch 6/20  loss: 2.2499  test acc: 0.194
Epoch 7/20  loss: 2.2313  test acc: 0.214
Epoch 8/20  loss: 2.2144  test acc: 0.224
Epoch 9/20  loss: 2.1971  test acc: 0.232
Epoch 10/20  loss: 2.1816  test acc: 0.244
Epoch 11/20  loss: 2.1655  test acc: 0.252
Epoch 12/20  loss: 2.1509  test acc: 0.254
Epoch 13/20  loss: 2.1366  test acc: 0.252
Epoch 14/20  loss: 2.1230  test acc: 0.264
Epoch 15/20  loss: 2.1099  test acc: 0.278
Epoch 16/20  loss: 2.0969  test acc: 0.280
Epoch 17/20  loss: 2.0846  test acc: 0.288
Epoch 18/20  loss: 2.0728  test acc: 0.292
Epoch 19/20  loss: 2.0613  test acc: 0.302
Epoch 20/20  loss: 2.0501  test acc: 0.306
